In [1]:
# pip install -U langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers

import os
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_classic.retrievers import MultiQueryRetriever


# ============================================================
# 1. Load Environment Variables
# ============================================================

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")


# ============================================================
# 2. Create Documents
# ============================================================

documents = [
    Document(
        page_content="Machine learning is a subset of artificial intelligence "
                      "that enables computers to learn from data."
    ),
    Document(
        page_content="Deep learning uses neural networks with multiple layers "
                      "to learn complex patterns from large datasets."
    ),
    Document(
        page_content="Natural Language Processing allows computers to understand "
                      "and process human language."
    ),
    Document(
        page_content="Generative AI can generate text, images, audio, video, "
                      "and other types of content."
    ),
    Document(
        page_content="Large Language Models are neural networks trained on "
                      "large amounts of text data."
    ),
]


# ============================================================
# 3. Create Embedding Model
# ============================================================

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


# ============================================================
# 4. Create FAISS Vector Store
# ============================================================

vectorstore = FAISS.from_documents(
    documents=documents,
    embedding=embeddings
)


# ============================================================
# 5. Create Normal Retriever
# ============================================================

base_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)


# ============================================================
# 6. Simple Hugging Face Model
# ============================================================

llm_endpoint = HuggingFaceEndpoint(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    task="text-generation",
    huggingfacehub_api_token=HF_TOKEN,
    max_new_tokens=128,
    temperature=0.1,
)

llm = ChatHuggingFace(
    llm=llm_endpoint
)


# ============================================================
# 7. Create MultiQuery Retriever
# ============================================================

retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm
)


# ============================================================
# 8. Query
# ============================================================

query = "What is AI that creates new content?"

docs = retriever.invoke(query)


# ============================================================
# 9. Display Results
# ============================================================

print("\nRetrieved Documents:\n")

for i, doc in enumerate(docs, start=1):
    print(f"--- Document {i} ---")
    print(doc.page_content)
    print()

c:\Users\arunk\anaconda3\envs\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\arunk\AppData\Local\Temp\ipykernel_26936\1950389799.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6026.13it/s]


BadRequestError: (Request ID: Root=1-6a7d6db6-516bcc9d5dfc22280adf15dc;ca592821-496d-4ac1-9131-398f3381f28e)

Bad request:
{'message': "The requested model 'HuggingFaceH4/zephyr-7b-beta' is not supported by any provider you have enabled.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}